In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier,
                               BaggingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, balanced_accuracy_score, fbeta_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")

# ===========================================================================
# 1. PATHS & DATA
# NO FE : "X_train_processed.csv" / "X_test_processed.csv"  → output: "tuned_results_no_fe.csv"
# WITH FE: "X_train_selected_imp.csv" / "X_test_selected_imp.csv" → output: "tuned_results_fe.csv"
# ===========================================================================
PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"
OUTPUTS_DIR   = PROJECT_ROOT / "outputs"
TABLE_DIR     = OUTPUTS_DIR / "comparison_tables"
FIG_DIR       = OUTPUTS_DIR / "plots"
for p in [MODELS_DIR, TABLE_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# *** CHANGE THESE 3 LINES ONLY ***
X_train      = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
X_test       = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
OUTPUT_CSV   = "tuned_results_fe.csv"       # tuned_results_no_fe.csv for without FE
PARAMS_CSV   = "best_hyperparameters_fe.csv"
PARAMS_JSON  = "best_hyperparameters_fe.json"

y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv    = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
print(f"Train: {X_train.shape} | Test: {X_test.shape} | Pos: {n_pos} Neg: {n_neg}")

# ===========================================================================
# 2. UTILITIES
# ===========================================================================
def safe_name(name):
    return name.lower().replace(" ", "_").replace("(","").replace(")","")

def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return model.predict(X).astype(float)

def composite_score(y_true, proba, preds):
    """
    Heavily weighted toward F2 (recall-focused) + AUC.
    This ensures tuned models improve recall without collapsing precision.
    """
    f2  = fbeta_score(y_true, preds, beta=2, zero_division=0)
    f1  = f1_score(y_true, preds, zero_division=0)
    try:
        auc = roc_auc_score(y_true, proba)
    except Exception:
        auc = 0.5
    rec = recall_score(y_true, preds, zero_division=0)
    # Penalize if recall < 0.70 (avoids Gaussian NB collapse)
    recall_penalty = 0.0 if rec >= 0.70 else (0.70 - rec) * 2.0
    return 0.35 * f2 + 0.35 * f1 + 0.30 * auc - recall_penalty

def find_best_threshold(y_true, proba):
    """Search threshold on F1, but enforce recall >= 0.70."""
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 901):
        preds = (proba >= t).astype(int)
        rec = recall_score(y_true, preds, zero_division=0)
        if rec < 0.70:          # hard floor on recall
            continue
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t

def cv_objective(trial, build_fn, X, y, cv):
    base_model = build_fn(trial)
    fold_scores = []
    for fold_i, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        model = clone(base_model)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        proba_val = predict_scores(model, X.iloc[va_idx])
        # Use threshold 0.4 during CV to favour recall
        preds_val = (proba_val >= 0.40).astype(int)
        fold_scores.append(composite_score(y.iloc[va_idx], proba_val, preds_val))
        trial.report(float(np.mean(fold_scores)), fold_i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_scores))

def run_study(name, build_fn, n_trials, X=None, y=None):
    if X is None: X = X_train
    if y is None: y = y_train
    print(f"Tuning {name} ({n_trials} trials)...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    study.optimize(lambda t: cv_objective(t, build_fn, X, y, cv),
                   n_trials=n_trials, show_progress_bar=False)
    best_model = build_fn(optuna.trial.FixedTrial(study.best_params))
    print(f"   Best CV score = {study.best_value:.4f}")
    return study, best_model

def run_study_no_prune(name, build_fn, n_trials, X=None, y=None):
    if X is None: X = X_train
    if y is None: y = y_train
    print(f"Tuning {name} ({n_trials} trials, no pruning)...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=optuna.pruners.NopPruner(),
    )
    study.optimize(lambda t: cv_objective(t, build_fn, X, y, cv),
                   n_trials=n_trials, show_progress_bar=False)
    best_model = build_fn(optuna.trial.FixedTrial(study.best_params))
    print(f"   Best CV score = {study.best_value:.4f}")
    return study, best_model

results_log     = {}
best_models     = {}
best_thresholds = {}
studies_store   = {}
best_params_log = {}

def finalize_model(name, model, cv_best_value=None, best_params=None):
    """
    FIX: threshold search on the SAME probability scale as final predictions.
    We fit the final model first, then search threshold on train OOF from
    a fresh clone — no calibration mismatch.
    """
    # Step 1: fit calibrated model (this is what will predict on test)
    calibrated = CalibratedClassifierCV(estimator=clone(model), method="sigmoid", cv=3)
    calibrated.fit(X_train, y_train)

    # Step 2: OOF probabilities from the CALIBRATED model for threshold search
    # Use cross_val_predict on the base model clone, then find threshold
    raw_clone = clone(model)
    try:
        oof_proba = cross_val_predict(raw_clone, X_train, y_train,
                                      cv=cv, method="predict_proba")[:, 1]
    except Exception:
        # fallback for models that don't support predict_proba in CV
        oof_proba = cross_val_predict(raw_clone, X_train, y_train,
                                      cv=cv, method="decision_function")
        oof_proba = (oof_proba - oof_proba.min()) / (oof_proba.max() - oof_proba.min() + 1e-12)

    best_t = find_best_threshold(y_train, oof_proba)

    # Step 3: test predictions
    test_proba  = predict_scores(calibrated, X_test)
    final_preds = (test_proba >= best_t).astype(int)

    # Sanity check — if recall collapsed, try 0.4 threshold
    rec = recall_score(y_test, final_preds, zero_division=0)
    if rec < 0.65:
        print(f"   ⚠ Recall {rec:.3f} too low at t={best_t:.3f}, falling back to t=0.40")
        best_t      = 0.40
        final_preds = (test_proba >= best_t).astype(int)

    results_log[name] = {
        "cv_score":          cv_best_value,
        "optimal_threshold": best_t,
        "accuracy":          accuracy_score(y_test, final_preds),
        "precision":         precision_score(y_test, final_preds, zero_division=0),
        "recall":            recall_score(y_test, final_preds, zero_division=0),
        "f1":                f1_score(y_test, final_preds, zero_division=0),
        "roc_auc":           roc_auc_score(y_test, test_proba),
    }
    best_thresholds[name] = best_t
    best_params_log[name] = best_params if best_params is not None else {}

    # Save with feature list
    joblib.dump(calibrated, MODELS_DIR / f"{safe_name(name)}_tuned_model.joblib")
    (MODELS_DIR / f"{safe_name(name)}_tuned_model_features.json").write_text(
        json.dumps(list(X_train.columns))
    )
    print(f"   Threshold={best_t:.3f} | Acc={results_log[name]['accuracy']:.4f} "
          f"| Rec={results_log[name]['recall']:.4f} | F1={results_log[name]['f1']:.4f} "
          f"| AUC={results_log[name]['roc_auc']:.4f}")

# ===========================================================================
# 3. SEARCH SPACES  (unchanged from your original — these are fine)
# ===========================================================================
def build_lr(trial):
    return LogisticRegression(
        C=trial.suggest_float("C", 1e-3, 1e3, log=True),
        penalty=trial.suggest_categorical("penalty", ["l1", "l2"]),
        solver="saga", max_iter=10000,
        tol=trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        class_weight="balanced", random_state=42
    )

def build_rf(trial):
    return RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 600),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_float("max_features", 0.3, 0.9),
        class_weight="balanced", random_state=42, n_jobs=-1
    )

def build_svc(trial):
    return SVC(
        C=trial.suggest_float("C", 1e-3, 50.0, log=True),
        kernel=trial.suggest_categorical("kernel", ["rbf", "linear"]),
        gamma="scale", class_weight="balanced",
        probability=True, random_state=42
    )

def build_gnb(trial):
    return GaussianNB(
        var_smoothing=trial.suggest_float("var_smoothing", 1e-12, 1e-6, log=True)
    )

def build_bagging(trial):
    return BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=trial.suggest_int("base_depth", 2, 20),
            min_samples_leaf=trial.suggest_int("base_min_leaf", 1, 10),
            class_weight="balanced", random_state=42
        ),
        n_estimators=trial.suggest_int("n_estimators", 50, 700),
        max_samples=trial.suggest_float("max_samples", 0.4, 1.0),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        random_state=42
    )

def build_gb(trial):
    return GradientBoostingClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),  # raised floor from 0.005
        max_depth=trial.suggest_int("max_depth", 2, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        max_features=trial.suggest_float("max_features", 0.4, 0.9),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        random_state=42
    )

def build_knn(trial):
    return KNeighborsClassifier(
        n_neighbors=trial.suggest_int("n_neighbors", 3, 30),
        weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
        metric=trial.suggest_categorical("metric", ["euclidean", "manhattan", "chebyshev"]),
        leaf_size=trial.suggest_int("leaf_size", 10, 60),
    )

def build_lda(trial):
    return LinearDiscriminantAnalysis(
        solver=trial.suggest_categorical("solver", ["lsqr", "eigen"]),
        shrinkage=trial.suggest_float("shrinkage", 0.0, 1.0),
        tol=trial.suggest_float("tol", 1e-6, 1e-2, log=True),
    )

def build_qda(trial):
    return QuadraticDiscriminantAnalysis(
        reg_param=trial.suggest_float("reg_param", 0.0, 1.0)
    )

def build_perceptron(trial):
    base = Perceptron(
        alpha=trial.suggest_float("alpha", 1e-6, 1.0, log=True),
        penalty=trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"]),
        l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),
        max_iter=trial.suggest_int("max_iter", 500, 3000),
        eta0=trial.suggest_float("eta0", 1e-4, 10.0, log=True),
        class_weight="balanced", random_state=42
    )
    return CalibratedClassifierCV(base, cv=3)

def build_dt(trial):
    return DecisionTreeClassifier(
        max_depth=trial.suggest_int("max_depth", 2, 15),          # capped at 15 (was 25 — overfitting)
        min_samples_split=trial.suggest_int("min_samples_split", 2, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 2, 20),  # floor 2 (was 1)
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy"]),  # removed log_loss
        ccp_alpha=trial.suggest_float("ccp_alpha", 1e-4, 0.05, log=True),      # tighter range
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        class_weight="balanced", random_state=42
    )

def build_ada(trial):
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=trial.suggest_int("base_depth", 1, 6),
            min_samples_leaf=trial.suggest_int("base_min_samples_leaf", 1, 15),
            class_weight="balanced", random_state=42
        ),
        n_estimators=trial.suggest_int("n_estimators", 50, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.001, 2.0, log=True),
        random_state=42
    )

def build_xgb(trial):
    scale_w = float(n_neg) / float(n_pos + 1e-5)
    return XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 700),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 10),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        scale_pos_weight=trial.suggest_float("scale_pos_weight", 1.0, scale_w * 1.8),
        random_state=42, n_jobs=-1, eval_metric="logloss"
    )

# ===========================================================================
# 4. TRAIN ALL MODELS
# ===========================================================================
models_to_train = [
    ("Logistic Regression", build_lr,        100, run_study),
    ("Decision Tree",       build_dt,        120, run_study_no_prune),
    ("Random Forest",       build_rf,         35, run_study),
    ("SVM",                 build_svc,        30, run_study),
    ("Gaussian NB",         build_gnb,        15, run_study),
    ("Bagging",             build_bagging,    60, run_study_no_prune),
    ("AdaBoost",            build_ada,       150, run_study_no_prune),
    ("Gradient Boosting",   build_gb,         35, run_study),
    ("KNN",                 build_knn,       100, run_study),
    ("LDA",                 build_lda,        50, run_study),
    ("QDA",                 build_qda,        15, run_study),
    ("Perceptron",          build_perceptron, 60, run_study),
]
if XGBOOST_AVAILABLE:
    models_to_train.append(("XGBoost", build_xgb, 90, run_study_no_prune))

for name, build_fn, trials, runner in models_to_train:
    try:
        study, base_model = runner(name, build_fn, n_trials=trials)
        best_models[name]   = base_model
        studies_store[name] = study
        finalize_model(name, base_model,
                       cv_best_value=study.best_value,
                       best_params=study.best_params)
        print(f"✓ {name} done.\n")
    except Exception as e:
        print(f"✗ Failed {name}: {e}")

# ===========================================================================
# 5. STACKING
# ===========================================================================
print("\nBuilding Stacking Meta-Learner...")
estimators_list = [(safe_name(n), clone(m)) for n, m in best_models.items()]

if len(estimators_list) >= 2:
    stacking_clf = StackingClassifier(
        estimators=estimators_list,
        final_estimator=LogisticRegression(C=1.0, class_weight="balanced",
                                           random_state=42, max_iter=2000),
        cv=5, n_jobs=-1, passthrough=False
    )
    finalize_model("Stacking ML", stacking_clf,
                   cv_best_value=None, best_params={"final_estimator__C": 1.0})

# ===========================================================================
# 6. RESULTS TABLE
# ===========================================================================
tuned_results = (pd.DataFrame.from_dict(results_log, orient="index")
                   .reset_index()
                   .rename(columns={"index": "Model"}))
tuned_results["Rank"] = tuned_results["f1"].rank(ascending=False, method="min").astype(int)
tuned_results = tuned_results.sort_values(["Rank", "roc_auc"], ascending=[True, False])
tuned_results.to_csv(TABLE_DIR / OUTPUT_CSV, index=False)
display(tuned_results)

# ===========================================================================
# 7. SAVE HYPERPARAMETERS
# ===========================================================================
best_params_rows = []
for name, params in best_params_log.items():
    for k, v in params.items():
        best_params_rows.append({"Model": name, "Hyperparameter": k, "Best Value": v})

best_params_df = pd.DataFrame(best_params_rows)
best_params_df.to_csv(TABLE_DIR / PARAMS_CSV, index=False)
with open(TABLE_DIR / PARAMS_JSON, "w") as f:
    json.dump(best_params_log, f, indent=2, default=str)
display(best_params_df)

Train: (378, 11) | Test: (163, 11) | Pos: 124 Neg: 254
Tuning Logistic Regression (100 trials)...
   Best CV score = 0.8883
   Threshold=0.516 | Acc=0.9202 | Rec=0.8302 | F1=0.8713 | AUC=0.9540
✓ Logistic Regression done.

Tuning Decision Tree (120 trials, no pruning)...
   Best CV score = 0.8569
   ⚠ Recall 0.453 too low at t=0.734, falling back to t=0.40
   Threshold=0.400 | Acc=0.8650 | Rec=0.8113 | F1=0.7963 | AUC=0.9101
✓ Decision Tree done.

Tuning Random Forest (35 trials)...
   Best CV score = 0.8959
   Threshold=0.419 | Acc=0.8712 | Rec=0.7925 | F1=0.8000 | AUC=0.9463
✓ Random Forest done.

Tuning SVM (30 trials)...
   Best CV score = 0.9012
   Threshold=0.399 | Acc=0.9202 | Rec=0.9057 | F1=0.8807 | AUC=0.9492
✓ SVM done.

Tuning Gaussian NB (15 trials)...
   Best CV score = 0.8927
   Threshold=0.839 | Acc=0.8773 | Rec=0.6792 | F1=0.7826 | AUC=0.9516
✓ Gaussian NB done.

Tuning Bagging (60 trials, no pruning)...
   Best CV score = 0.9045
   Threshold=0.404 | Acc=0.8896 | Rec=0

,Model,cv_score,optimal_threshold,accuracy,precision,recall,f1,roc_auc,Rank
3,SVM,0.901215,0.399,0.920245,0.857143,0.905660,0.880734,0.949228,1
11,Perceptron,0.892111,0.397,0.920245,0.900000,0.849057,0.873786,0.953173,2
0,Logistic Regression,0.888306,0.516,0.920245,0.916667,0.830189,0.871287,0.954031,3
9,LDA,0.894373,0.377,0.907975,0.851852,0.867925,0.859813,0.950429,4
12,XGBoost,0.898996,0.451,0.907975,0.880000,0.830189,0.854369,0.954374,5
10,QDA,0.887217,0.386,0.901840,0.849057,0.849057,0.849057,0.947513,6
7,Gradient Boosting,0.891181,0.393,0.901840,0.877551,0.811321,0.843137,0.952830,7
13,Stacking ML,NaN,0.641,0.901840,0.911111,0.773585,0.836735,0.956089,8
5,Bagging,0.904532,0.404,0.889571,0.830189,0.830189,0.830189,0.949914,9
6,AdaBoost,0.899996,0.402,0.883436,0.840000,0.792453,0.815534,0.950429,10


,Model,Hyperparameter,Best Value
0,Logistic Regression,C,0.1363
1,Logistic Regression,penalty,l1
2,Logistic Regression,tol,0.001728
3,Decision Tree,max_depth,12
4,Decision Tree,min_samples_split,7
5,Decision Tree,min_samples_leaf,3
6,Decision Tree,criterion,gini
7,Decision Tree,ccp_alpha,0.000396
8,Decision Tree,max_features,sqrt
9,Random Forest,n_estimators,51


In [1]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                               GradientBoostingClassifier, RandomForestClassifier,
                               StackingClassifier)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score,
    roc_auc_score, average_precision_score, confusion_matrix, log_loss,
    RocCurveDisplay
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:
    XGBOOST_AVAILABLE = False
    print("XGBoost unavailable:", exc)

sns.set_theme(style="whitegrid")

PROJECT_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"

BASE_OUT  = PROJECT_ROOT / "outputs" / "with_FE_tuned"
TABLE_DIR = PROJECT_ROOT / "outputs" / "comparison_tables"

for p in [MODELS_DIR, TABLE_DIR, BASE_OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Output root:", BASE_OUT)

Output root: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned


In [2]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_selected_imp.csv")
X_test  = pd.read_csv(PROCESSED_DIR / "X_test_selected_imp.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["PCOS"]
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv")["PCOS"]

cv    = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())

print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Pos: {n_pos}  Neg: {n_neg}")

Train : (378, 11) | Test : (163, 11)
Pos: 124  Neg: 254


In [3]:
def safe_name(name):
    return (name.lower()
            .replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", "")
            .replace("+", "plus").replace(":", ""))


def model_dir(model_name):
    d = BASE_OUT / safe_name(model_name)
    d.mkdir(parents=True, exist_ok=True)
    return d


def predict_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-12)
    return model.predict(X).astype(float)


def composite_score(y_true, proba, preds):
    f2  = fbeta_score(y_true, preds, beta=2, zero_division=0)
    f1  = f1_score(y_true, preds, zero_division=0)
    try:
        auc = roc_auc_score(y_true, proba)
    except Exception:
        auc = 0.5
    rec = recall_score(y_true, preds, zero_division=0)
    recall_penalty = 0.0 if rec >= 0.70 else (0.70 - rec) * 2.0
    return 0.35 * f2 + 0.35 * f1 + 0.30 * auc - recall_penalty


def find_best_threshold(y_true, proba):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 901):
        preds = (proba >= t).astype(int)
        if recall_score(y_true, preds, zero_division=0) < 0.70:
            continue
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t


def cv_objective(trial, build_fn, X, y, cv):
    base_model = build_fn(trial)
    fold_scores = []
    for fold_i, (tr_idx, va_idx) in enumerate(cv.split(X, y)):
        model = clone(base_model)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        proba_val = predict_scores(model, X.iloc[va_idx])
        preds_val = (proba_val >= 0.40).astype(int)
        fold_scores.append(composite_score(y.iloc[va_idx], proba_val, preds_val))
        trial.report(float(np.mean(fold_scores)), fold_i)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(fold_scores))


def run_study(name, build_fn, n_trials):
    print(f"  Tuning {name} ({n_trials} trials)...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    study.optimize(lambda t: cv_objective(t, build_fn, X_train, y_train, cv),
                   n_trials=n_trials, show_progress_bar=False)
    print(f"  Best CV score = {study.best_value:.4f}")
    return study, build_fn(optuna.trial.FixedTrial(study.best_params))


def run_study_no_prune(name, build_fn, n_trials):
    print(f"  Tuning {name} ({n_trials} trials, no pruning)...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42, multivariate=True),
        pruner=optuna.pruners.NopPruner(),
    )
    study.optimize(lambda t: cv_objective(t, build_fn, X_train, y_train, cv),
                   n_trials=n_trials, show_progress_bar=False)
    print(f"  Best CV score = {study.best_value:.4f}")
    return study, build_fn(optuna.trial.FixedTrial(study.best_params))

print("Utility functions defined.")

Utility functions defined.


In [ ]:
def plot_confusion(y_true, preds, model_name, out_dir):
    cm = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No", "Yes"], yticklabels=["No", "Yes"], ax=ax)
    ax.set_title(f"with FE Tuned — {model_name}\nConfusion Matrix",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    path = out_dir / "confusion_matrix.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


# ── Plot: ROC Curve ────────────────────────────────────────────────────────
def plot_roc(y_true, scores, model_name, out_dir):
    auc = roc_auc_score(y_true, scores)
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # 1. Pass an empty or generic string to name to prevent standard concatenation conflicts
    disp = RocCurveDisplay.from_predictions(y_true, scores, name="", ax=ax)
    
    # 2. Directly overwrite the line label with your precise 4-decimal string
    disp.line_.set_label(f"Classifier (AUC = {auc:.4f})")
    
    # 3. Re-draw the legend to apply your custom label formatting
    ax.legend(loc="lower right")
    
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_title(f"with FE Tuned — {model_name}\nROC Curve", fontsize=11, fontweight="bold")
    plt.tight_layout()
    path = out_dir / "roc_curve.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")



def plot_stratified_cv(model, model_name, out_dir):
    fold_acc, fold_f1, fold_auc = [], [], []
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    for tr, va in skf.split(X_train, y_train):
        m = clone(model)
        m.fit(X_train.iloc[tr], y_train.iloc[tr])
        sc = predict_scores(m, X_train.iloc[va])
        th = find_best_threshold(y_train.iloc[tr],
                                  predict_scores(m, X_train.iloc[tr]))
        preds = (sc >= th).astype(int)
        fold_acc.append(accuracy_score(y_train.iloc[va], preds))
        fold_f1.append(f1_score(y_train.iloc[va], preds, zero_division=0))
        try:
            fold_auc.append(roc_auc_score(y_train.iloc[va], sc))
        except Exception:
            fold_auc.append(np.nan)

    folds = np.arange(1, 11)
    fig, ax = plt.subplots(figsize=(12, 5))
    for vals, label, marker in [
        (fold_acc, f"Accuracy (mean={np.nanmean(fold_acc):.4f})", "o"),
        (fold_f1,  f"F1       (mean={np.nanmean(fold_f1):.4f})",  "s"),
        (fold_auc, f"ROC-AUC  (mean={np.nanmean(fold_auc):.4f})", "^"),
    ]:
        ax.plot(folds, vals, marker=marker, linewidth=1.8, label=label)
        for x, y in zip(folds, vals):
            ax.annotate(f"{y:.4f}", (x, y),
                        textcoords="offset points", xytext=(0, 6),
                        ha="center", fontsize=7.5, color="dimgray")

    ax.set_xticks(folds)
    ax.set_xticklabels([f"Fold {i}" for i in folds])
    ax.set_ylabel("Score")
    ax.set_ylim(max(0, min(fold_acc + fold_f1 + fold_auc) - 0.08), 1.08)
    ax.set_title(
        f"with FE Tuned — {model_name}\n"
        f"10-Fold Stratified CV ({X_train.shape[1]} Features)",
        fontsize=11, fontweight="bold"
    )
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    path = out_dir / "stratified_cv.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


def plot_hyperparam_curves(study, model_name, out_dir):
    """
    One clean PNG per hyperparameter, saved inside out_dir/hyperparams/.
    Continuous params are binned into 10 buckets for readability.
    """
    trials_df = study.trials_dataframe(attrs=("params", "value"))
    trials_df = trials_df.dropna(subset=["value"])

    param_cols = [c for c in trials_df.columns if c.startswith("params_")]
    if not param_cols:
        print(f"  No hyperparameter data found for {model_name}")
        return

    # Save each param into its own subfolder
    hp_dir = out_dir / "hyperparams"
    hp_dir.mkdir(parents=True, exist_ok=True)

    for col in param_cols:
        param_name = col.replace("params_", "")
        sub = trials_df[[col, "value"]].dropna().copy()
        if sub.empty or sub[col].nunique() < 2:
            continue

        # ── Bin continuous floats into 10 buckets ──────────────────────
        is_continuous = (sub[col].dtype in [np.float64, np.float32] and
                         sub[col].nunique() > 15)

        if is_continuous:
            sub["bin"] = pd.cut(sub[col], bins=10)
            grp   = sub.groupby("bin")["value"]
            means = grp.mean()
            stds  = grp.std().fillna(0)
            # Use bin midpoint as x-label
            x_labels = [f"{iv.mid:.3f}" for iv in means.index]
            x_pos    = np.arange(len(x_labels))
        else:
            grp      = sub.groupby(col)["value"]
            means    = grp.mean()
            stds     = grp.std().fillna(0)
            x_labels = [str(v) for v in means.index]
            x_pos    = np.arange(len(x_labels))

        best_idx  = means.values.argmax()
        best_mean = means.values[best_idx]
        best_label= x_labels[best_idx]

        # ── Plot ───────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 4))

        ax.errorbar(x_pos, means.values, yerr=stds.values,
                    fmt="o-", color="#1f77b4", capsize=5,
                    linewidth=2, markersize=6, elinewidth=1.2)

        ax.scatter([x_pos[best_idx]], [best_mean],
                   color="red", zorder=6, s=80,
                   label=f"best: {best_label}  (score={best_mean:.4f})")

        ax.set_xticks(x_pos)
        ax.set_xticklabels(x_labels, rotation=30, ha="right", fontsize=9)
        ax.set_xlabel(param_name, fontsize=10)
        ax.set_ylabel("Mean CV Score", fontsize=10)
        ax.set_title(
            f"with FE Tuned — {model_name}\n"
            f"CV Score vs  {param_name}",
            fontsize=11, fontweight="bold"
        )
        ax.legend(fontsize=9, loc="best")
        ax.yaxis.set_major_formatter(plt.FormatStrFormatter("%.4f"))
        ax.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()

        safe_param = param_name.replace("/", "_").replace(" ", "_")
        path = hp_dir / f"hyperparam_{safe_param}.png"
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved → {path}")

print("Plot functions defined.")

Plot functions defined.


In [5]:
def build_lr(trial):
    return LogisticRegression(
        C=trial.suggest_float("C", 1e-3, 1e3, log=True),
        penalty=trial.suggest_categorical("penalty", ["l1", "l2"]),
        solver="saga", max_iter=10000,
        tol=trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        class_weight="balanced", random_state=42)

def build_rf(trial):
    return RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 600),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_float("max_features", 0.3, 0.9),
        class_weight="balanced", random_state=42, n_jobs=-1)

def build_svc(trial):
    return SVC(
        C=trial.suggest_float("C", 1e-3, 50.0, log=True),
        kernel=trial.suggest_categorical("kernel", ["rbf", "linear"]),
        gamma="scale", class_weight="balanced",
        probability=True, random_state=42)

def build_gnb(trial):
    return GaussianNB(
        var_smoothing=trial.suggest_float("var_smoothing", 1e-12, 1e-6, log=True))

def build_bagging(trial):
    return BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=trial.suggest_int("base_depth", 2, 20),
            min_samples_leaf=trial.suggest_int("base_min_leaf", 1, 10),
            class_weight="balanced", random_state=42),
        n_estimators=trial.suggest_int("n_estimators", 50, 700),
        max_samples=trial.suggest_float("max_samples", 0.4, 1.0),
        max_features=trial.suggest_float("max_features", 0.3, 1.0),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        random_state=42)

def build_gb(trial):
    return GradientBoostingClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 500),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        max_features=trial.suggest_float("max_features", 0.4, 0.9),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        random_state=42)

def build_knn(trial):
    return KNeighborsClassifier(
        n_neighbors=trial.suggest_int("n_neighbors", 3, 30),
        weights=trial.suggest_categorical("weights", ["uniform", "distance"]),
        metric=trial.suggest_categorical("metric", ["euclidean", "manhattan", "chebyshev"]),
        leaf_size=trial.suggest_int("leaf_size", 10, 60))

def build_lda(trial):
    return LinearDiscriminantAnalysis(
        solver=trial.suggest_categorical("solver", ["lsqr", "eigen"]),
        shrinkage=trial.suggest_float("shrinkage", 0.0, 1.0),
        tol=trial.suggest_float("tol", 1e-6, 1e-2, log=True))

def build_qda(trial):
    return QuadraticDiscriminantAnalysis(
        reg_param=trial.suggest_float("reg_param", 0.0, 1.0))

def build_perceptron(trial):
    base = Perceptron(
        alpha=trial.suggest_float("alpha", 1e-6, 1.0, log=True),
        penalty=trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"]),
        l1_ratio=trial.suggest_float("l1_ratio", 0.0, 1.0),
        max_iter=trial.suggest_int("max_iter", 500, 3000),
        eta0=trial.suggest_float("eta0", 1e-4, 10.0, log=True),
        class_weight="balanced", random_state=42)
    return CalibratedClassifierCV(base, cv=3)

def build_dt(trial):
    return DecisionTreeClassifier(
        max_depth=trial.suggest_int("max_depth", 2, 15),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 2, 20),
        criterion=trial.suggest_categorical("criterion", ["gini", "entropy"]),
        ccp_alpha=trial.suggest_float("ccp_alpha", 1e-4, 0.05, log=True),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        class_weight="balanced", random_state=42)

def build_ada(trial):
    return AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=trial.suggest_int("base_depth", 1, 6),
            min_samples_leaf=trial.suggest_int("base_min_samples_leaf", 1, 15),
            class_weight="balanced", random_state=42),
        n_estimators=trial.suggest_int("n_estimators", 50, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.001, 2.0, log=True),
        random_state=42)

def build_xgb(trial):
    scale_w = float(n_neg) / float(n_pos + 1e-5)
    return XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 50, 700),
        learning_rate=trial.suggest_float("learning_rate", 0.003, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 10),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 5.0, log=True),
        scale_pos_weight=trial.suggest_float("scale_pos_weight", 1.0, scale_w * 1.8),
        random_state=42, n_jobs=-1, eval_metric="logloss")

print("Search spaces defined.")

Search spaces defined.


In [6]:
results_log     = {}
best_models     = {}
best_thresholds = {}
studies_store   = {}
best_params_log = {}

models_to_train = [
    ("Logistic Regression", build_lr,         100, run_study),
    ("Decision Tree",       build_dt,         120, run_study_no_prune),
    ("Random Forest",       build_rf,          35, run_study),
    ("SVM",                 build_svc,         30, run_study),
    ("Gaussian NB",         build_gnb,         15, run_study),
    ("Bagging",             build_bagging,     60, run_study_no_prune),
    ("AdaBoost",            build_ada,        150, run_study_no_prune),
    ("Gradient Boosting",   build_gb,          35, run_study),
    ("KNN",                 build_knn,        100, run_study),
    ("LDA",                 build_lda,         50, run_study),
    ("QDA",                 build_qda,         15, run_study),
    ("Perceptron",          build_perceptron,  60, run_study),
]
if XGBOOST_AVAILABLE:
    models_to_train.append(("XGBoost", build_xgb, 90, run_study_no_prune))

for name, build_fn, trials, runner in models_to_train:
    print(f"\n▶ {name}")
    try:
        t_tune_start = time.time()
        study, base_model = runner(name, build_fn, n_trials=trials)
        tune_time_sec = round(time.time() - t_tune_start, 4)
        out_dir = model_dir(name)

        # ── Calibrate & finalize ──────────────────────────────────────
        calibrated = CalibratedClassifierCV(
            estimator=clone(base_model), method="sigmoid", cv=3)
        t_train_start = time.time()
        calibrated.fit(X_train, y_train)
        train_time_sec = round(time.time() - t_train_start, 4)

        raw_clone = clone(base_model)
        try:
            oof_proba = cross_val_predict(raw_clone, X_train, y_train,
                                          cv=cv, method="predict_proba")[:, 1]
        except Exception:
            oof_proba = cross_val_predict(raw_clone, X_train, y_train,
                                          cv=cv, method="decision_function")
            oof_proba = ((oof_proba - oof_proba.min()) /
                         (oof_proba.max() - oof_proba.min() + 1e-12))

        best_t = find_best_threshold(y_train, oof_proba)
        test_proba  = predict_scores(calibrated, X_test)
        final_preds = (test_proba >= best_t).astype(int)

        if recall_score(y_test, final_preds, zero_division=0) < 0.65:
            print(f"  ⚠ Recall too low, falling back to t=0.40")
            best_t      = 0.40
            final_preds = (test_proba >= best_t).astype(int)

        results_log[name] = {
            "cv_score"          : study.best_value,
            "optimal_threshold" : best_t,
            "accuracy"          : accuracy_score(y_test, final_preds),
            "precision"         : precision_score(y_test, final_preds, zero_division=0),
            "recall"            : recall_score(y_test, final_preds, zero_division=0),
            "f1"                : f1_score(y_test, final_preds, zero_division=0),
            "roc_auc"           : roc_auc_score(y_test, test_proba),
            "tune_time_sec"     : tune_time_sec,
            "train_time_sec"    : train_time_sec,
            "total_time_sec"    : round(tune_time_sec + train_time_sec, 4),
        }
        best_thresholds[name] = best_t
        best_models[name]     = base_model
        studies_store[name]   = study
        best_params_log[name] = study.best_params

        # ── Plots ─────────────────────────────────────────────────────
        plot_confusion(y_test,    final_preds, name, out_dir)
        plot_roc(y_test,          test_proba,  name, out_dir)
        plot_stratified_cv(base_model,         name, out_dir)
        plot_hyperparam_curves(study,          name, out_dir)

        # ── Save model ─────────────────────────────────────────────────
        joblib.dump(calibrated,
                    MODELS_DIR / f"with_fe_tuned_{safe_name(name)}.joblib")
        (MODELS_DIR / f"with_fe_tuned_{safe_name(name)}_features.json").write_text(
            json.dumps(list(X_train.columns)))

        print(f"  ✓ t={best_t:.3f} | "
              f"Acc={results_log[name]['accuracy']:.4f} | "
              f"Rec={results_log[name]['recall']:.4f} | "
              f"F1={results_log[name]['f1']:.4f} | "
              f"AUC={results_log[name]['roc_auc']:.4f} | "
              f"Time={train_time_sec:.2f}s")
    except Exception as e:
        print(f"  ✗ Failed: {e}")


▶ Logistic Regression
  Tuning Logistic Regression (100 trials)...
  Best CV score = 0.8883
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\logistic_regression\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\logistic_regression\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\logistic_regression\stratified_cv.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\logistic_regression\hyperparams\hyperparam_C.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\logistic_regression\hyperparams\hyperparam_penalty.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\

In [7]:
print("\n▶ Building Stacking Meta-Learner...")
estimators_list = [(safe_name(n), clone(m)) for n, m in best_models.items()]

if len(estimators_list) >= 2:
    stacking_clf = StackingClassifier(
        estimators=estimators_list,
        final_estimator=LogisticRegression(
            C=1.0, class_weight="balanced", random_state=42, max_iter=2000),
        cv=5, n_jobs=-1, passthrough=False
    )
    out_dir = model_dir("Stacking ML")

    calibrated_stack = CalibratedClassifierCV(
        estimator=clone(stacking_clf), method="sigmoid", cv=3)
    t_train_start = time.time()
    calibrated_stack.fit(X_train, y_train)
    train_time_sec = round(time.time() - t_train_start, 4)

    try:
        oof_proba = cross_val_predict(clone(stacking_clf), X_train, y_train,
                                      cv=cv, method="predict_proba")[:, 1]
    except Exception:
        oof_proba = predict_scores(calibrated_stack, X_train)

    best_t     = find_best_threshold(y_train, oof_proba)
    test_proba = predict_scores(calibrated_stack, X_test)
    final_preds= (test_proba >= best_t).astype(int)

    if recall_score(y_test, final_preds, zero_division=0) < 0.65:
        best_t      = 0.40
        final_preds = (test_proba >= best_t).astype(int)

    results_log["Stacking ML"] = {
        "cv_score"          : None,
        "optimal_threshold" : best_t,
        "accuracy"          : accuracy_score(y_test, final_preds),
        "precision"         : precision_score(y_test, final_preds, zero_division=0),
        "recall"            : recall_score(y_test, final_preds, zero_division=0),
        "f1"                : f1_score(y_test, final_preds, zero_division=0),
        "roc_auc"           : roc_auc_score(y_test, test_proba),
        "train_time_sec"    : train_time_sec,
    }
    best_params_log["Stacking ML"] = {"final_estimator__C": 1.0}

    plot_confusion(y_test,    final_preds,      "Stacking ML", out_dir)
    plot_roc(y_test,          test_proba,       "Stacking ML", out_dir)
    plot_stratified_cv(stacking_clf,            "Stacking ML", out_dir)

    joblib.dump(calibrated_stack,
                MODELS_DIR / "with_fe_tuned_stacking_ml.joblib")
    (MODELS_DIR / "with_fe_tuned_stacking_ml_features.json").write_text(
        json.dumps(list(X_train.columns)))

    print(f"  ✓ Stacking done | "
          f"Acc={results_log['Stacking ML']['accuracy']:.4f} | "
          f"AUC={results_log['Stacking ML']['roc_auc']:.4f}")


▶ Building Stacking Meta-Learner...
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\stacking_ml\confusion_matrix.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\stacking_ml\roc_curve.png
  Saved → c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs\with_FE_tuned\stacking_ml\stratified_cv.png
  ✓ Stacking done | Acc=0.9018 | AUC=0.9561


In [ ]:
'''tuned_df = (pd.DataFrame.from_dict(results_log, orient="index")
              .reset_index()
              .rename(columns={"index": "Model"}))
tuned_df["Rank"] = tuned_df["f1"].rank(ascending=False, method="min").astype(int)
tuned_df = tuned_df.sort_values(["Rank", "roc_auc"], ascending=[True, False])
tuned_df.to_csv(TABLE_DIR / "with_FE_tuned_results.csv", index=False)
display(tuned_df)

rows = []
for mdl_name, params in best_params_log.items():
    for k, v in params.items():
        rows.append({"Model": mdl_name, "Hyperparameter": k, "Best Value": v})

params_df = pd.DataFrame(rows)
params_df.to_csv(TABLE_DIR / "best_hyperparameters_with_fe.csv", index=False)

import json as _json
with open(TABLE_DIR / "best_hyperparameters_with_fe.json", "w") as f:
    _json.dump(best_params_log, f, indent=2, default=str)

display(params_df)
print("\nAll done. Outputs saved under:", BASE_OUT)'''
from pathlib import Path
import pandas as pd
import json

# 1. Define explicit base directory dynamically
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUT = PROJECT_ROOT / "outputs"
BASE_OUT.mkdir(parents=True, exist_ok=True)

# Ensure directories exist
BASE_OUT.mkdir(parents=True, exist_ok=True)

# 2. Build results DataFrame using the exact keys visible in your terminal table output
tuned_df = (pd.DataFrame.from_dict(results_log, orient="index")
            .reset_index()
            .rename(columns={"index": "Model"}))

# Rank and sort exactly as shown in your script code
tuned_df["Rank"] = tuned_df["f1"].rank(ascending=False, method="min").astype(int)
tuned_df = tuned_df.sort_values(["Rank", "roc_auc"], ascending=[True, False])

# 3. Save the results tracking sheets to your actual path
tuned_df.to_csv(TABLE_DIR / "with_FE_tuned_results.csv", index=False)

# 4. Save your hyperparameters log
rows = []
for mdl_name, params in best_params_log.items():
    for k, v in params.items():
        rows.append({"Model": mdl_name, "Hyperparameter": k, "Best Value": v})

params_df = pd.DataFrame(rows)
params_df.to_csv(TABLE_DIR / "best_hyperparameters_with_fe.csv", index=False)

# 5. Save the raw JSON configuration tracking file
with open(TABLE_DIR / "best_hyperparameters_with_fe.json", "w") as f:
    json.dump(best_params_log, f, indent=2, default=str)

# 6. Clean display using your active column names
display(tuned_df[[
    "Rank", "Model", "accuracy", "precision", "recall", 
    "f1", "roc_auc", "tune_time_sec", "train_time_sec", "total_time_sec"
]])

print("\nAll done. Outputs successfully saved under:", BASE_OUT)


,Rank,Model,accuracy,precision,recall,f1,roc_auc,tune_time_sec,train_time_sec,total_time_sec
3,1,SVM,0.920245,0.857143,0.905660,0.880734,0.949228,9.5851,0.0791,9.6642
11,2,Perceptron,0.920245,0.900000,0.849057,0.873786,0.953173,18.7838,0.0985,18.8823
0,3,Logistic Regression,0.920245,0.916667,0.830189,0.871287,0.954031,13.3807,0.0268,13.4075
9,4,LDA,0.907975,0.851852,0.867925,0.859813,0.950429,7.3443,0.0310,7.3753
12,5,XGBoost,0.907975,0.880000,0.830189,0.854369,0.954374,291.8697,1.5953,293.4650
10,6,QDA,0.901840,0.849057,0.849057,0.849057,0.947513,1.6194,0.0301,1.6495
7,7,Gradient Boosting,0.901840,0.877551,0.811321,0.843137,0.952830,79.9897,0.6867,80.6764
13,8,Stacking ML,0.901840,0.911111,0.773585,0.836735,0.956089,NaN,53.4301,NaN
5,9,Bagging,0.889571,0.830189,0.830189,0.830189,0.949914,800.4498,6.5126,806.9624
6,10,AdaBoost,0.883436,0.840000,0.792453,0.815534,0.950429,2168.7066,1.1393,2169.8459



All done. Outputs successfully saved under: c:\Users\ssath\OneDrive\Documents\PCOS detection using ML\pcos-prediction-ml-clean\outputs
